# 02. Data Cleaning
**Objective:** Execute the data cleaning action plan derived from `01_data_understanding.ipynb` to produce a verified, analysis-ready dataset for feature engineering and modeling.

### Cleaning Checklist:
- [x] Standardize sector typo (`"Commerical"` $\rightarrow$ `"Commercial"`)
- [x] Drop `Transportation` sector (100% unrecorded target values)
- [x] Drop partial year `2024` (preserve complete annual cycles: 1973–2023)
- [x] Fuse `Hydroelectric Power` & `Conventional Hydroelectric Power` $\rightarrow$ `hydro`
- [x] Clip physical boundary violation (`-0.002` $\rightarrow$ `0.0`)
- [x] Final validation (Sanity Checks) & Export to `data/processed/renewable_clean.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_PATH = Path('../data/raw/dataset.csv')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
raw_df = pd.read_csv(RAW_PATH)
df = raw_df.copy()

print(f"Initial raw shape : {df.shape[0]:,} rows x {df.shape[1]} columns")

Initial raw shape : 3,065 rows x 17 columns


In [4]:
df.head(5)

,Year,Month,Sector,Hydroelectric Power,Geothermal Energy,Solar Energy,Wind Energy,Wood Energy,Waste Energy,"Fuel Ethanol, Excluding Denaturant",Biomass Losses and Co-products,Biomass Energy,Total Renewable Energy,Renewable Diesel Fuel,Other Biofuels,Conventional Hydroelectric Power,Biodiesel
0,1973,1,Commerical,0.00,0.00,0.0,0.0,0.570,0.000,0.0,0.0,0.570,0.570,0.0,0.0,0.000,0.0
1,1973,1,Electric Power,0.00,0.49,0.0,0.0,0.054,0.157,0.0,0.0,0.211,89.223,0.0,0.0,88.522,0.0
2,1973,1,Industrial,1.04,0.00,0.0,0.0,98.933,0.000,0.0,0.0,98.933,99.973,0.0,0.0,0.000,0.0
3,1973,1,Residential,0.00,0.00,0.0,0.0,30.074,0.000,0.0,0.0,0.000,30.074,0.0,0.0,0.000,0.0
4,1973,1,Transportation,0.00,0.00,0.0,0.0,0.000,0.000,0.0,0.0,0.000,0.000,0.0,0.0,0.000,0.0


In [5]:
raw_df.Sector.value_counts()

Sector
Commerical        613
Electric Power    613
Industrial        613
Residential       613
Transportation    613
Name: count, dtype: int64

### 1. Standardize Sector Typo
**Objective:** Fix the spelling inconsistency (`"Commerical"` $\rightarrow$ `"Commercial"`) identified during the Data Health Audit.

In [6]:
df['Sector'] = df["Sector"].replace({"Commerical": "Commercial"})
df.Sector.unique()

array(['Commercial', 'Electric Power', 'Industrial', 'Residential',
       'Transportation'], dtype=object)

### 2. Remove Uninformative Sector (`Transportation`)
**Objective:** Drop the `Transportation` sector because 100% of its target records (`Total Renewable Energy`) are unrecorded (`0.0`).

In [7]:
rows_before = len(df)
df = df[df['Sector']!='Transportation'].copy()
rows_dropped = rows_before - len(df)
print(f"Dropped {rows_dropped} rows from Transportation Sector.")
print(f"Current shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Currenct Sectors : {df.Sector.unique()}")

Dropped 613 rows from Transportation Sector.
Current shape: 2,452 rows x 17 columns
Currenct Sectors : ['Commercial' 'Electric Power' 'Industrial' 'Residential']


### 3. Remove Partial Year (`2024`)
**Objective:** Filter out January `2024` (4 rows) to maintain complete 12-month annual cycles across `1973–2023`.

In [8]:
rows_before = len(df)
df = df[df['Year'] <=2023].copy()
rows_dropped = rows_before - len(df)

print(f"Dropped {rows_dropped} rows from partial year 2024.")
print(f"Current shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")

Dropped 4 rows from partial year 2024.
Current shape: 2,448 rows x 17 columns
Year range: 1973 – 2023


### 4. Clip Boundary Violation & Fuse Hydro Columns
**Objective:** Clip the single negative hydro value (`-0.002` in Aug 2007) to `0.0` before fusing the two mutually exclusive Hydro columns into `hydro`.

In [9]:
neg_before = (df["Hydroelectric Power"] < 0).sum()
anomaly_data = df[df["Hydroelectric Power"] < 0][
    ["Year", "Month", "Hydroelectric Power", "Total Renewable Energy"]
]
df["Hydroelectric Power"] = df["Hydroelectric Power"].clip(lower=0.0)
neg_after = (df["Hydroelectric Power"] < 0).sum()

df["hydro"] = df[["Conventional Hydroelectric Power", "Hydroelectric Power"]].max(
    axis=1
)

df = df.drop(columns=["Conventional Hydroelectric Power", "Hydroelectric Power"])
print(
    f"Fused into 'hydro'. Negative values before clipping: {neg_before} -> after: {neg_after}"
)
print(f"Current shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Fused into 'hydro'. Negative values before clipping: 1 -> after: 0
Current shape: 2,448 rows x 16 columns


In [10]:
print(f"Anomaly Data : {anomaly_data}")

Anomaly Data :       Year  Month  Hydroelectric Power  Total Renewable Energy
2075  2007      8               -0.002                   9.999


### Final Validation (Sanity Checks) & Export
**Objective:** Sort chronologically, enforce strict automated data-quality assertions, and export the verified dataset to `data/processed/renewable_clean.csv`.

In [11]:
df = df.sort_values(by=['Year','Month','Sector']).reset_index(drop=True)

expected_sectors = {'Commercial', 'Electric Power', 'Industrial', 'Residential'}

assert len(df) == 2448, f"Expected 2448 rows, got {len(df)}"
assert df.isna().sum().sum() == 0, "Dataset still contains NaN values!"
assert (df.select_dtypes(include=[np.number]) < 0).sum().sum() == 0, "Dataset still contains negative numbers!"
assert df.duplicated(subset=['Year', 'Month', 'Sector']).sum() == 0, "Duplicate composite keys found!"
assert set(df['Sector'].unique()) == expected_sectors, f"Unexpected sectors: {df['Sector'].unique()}"
assert (df['Total Renewable Energy'] == 0).sum() == 0, "Target variable still contains zero values!"
print("All 6 automated data-quality assertions PASSED!")

All 6 automated data-quality assertions PASSED!


In [12]:
CLEAN_PATH = PROCESSED_DIR / 'renewable_clean.csv'
df.to_csv(CLEAN_PATH, index=False)
print(f"Exported clean dataset to: {CLEAN_PATH}")
print(f"Final verified shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Exported clean dataset to: ..\data\processed\renewable_clean.csv
Final verified shape: 2,448 rows x 16 columns


,Year,Month,Sector,Geothermal Energy,Solar Energy,Wind Energy,Wood Energy,Waste Energy,"Fuel Ethanol, Excluding Denaturant",Biomass Losses and Co-products,Biomass Energy,Total Renewable Energy,Renewable Diesel Fuel,Other Biofuels,Biodiesel,hydro
0,1973,1,Commercial,0.00,0.0,0.0,0.570,0.000,0.0,0.0,0.570,0.570,0.0,0.0,0.0,0.000
1,1973,1,Electric Power,0.49,0.0,0.0,0.054,0.157,0.0,0.0,0.211,89.223,0.0,0.0,0.0,88.522
2,1973,1,Industrial,0.00,0.0,0.0,98.933,0.000,0.0,0.0,98.933,99.973,0.0,0.0,0.0,1.040
3,1973,1,Residential,0.00,0.0,0.0,30.074,0.000,0.0,0.0,0.000,30.074,0.0,0.0,0.0,0.000
4,1973,2,Commercial,0.00,0.0,0.0,0.515,0.000,0.0,0.0,0.515,0.515,0.0,0.0,0.0,0.000
